## FAIRe YAML to JSON

In [ ]:
import yaml
import json


def slots_to_json(yaml_file):
    with open(yaml_file, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

    result = {}

    for slot_name, slot in data.get("slots", {}).items():
        result[slot_name] = {
            "value": (
                slot.get("examples", [{}])[0].get("value")
                if slot.get("examples")
                else None
            ),
            "description": slot.get("description")
        }

    return result

result = slots_to_json("FAIRe_schema_demo_SSSOMatic_2026-10-01.yaml")

with open("faire_yaml_slot.json", "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

# DwC DP Table Schemas to JSON

In [ ]:
!git clone https://github.com/gbif/dwc-dp.git
!cp -r dwc-dp/dwc-dp/table-schemas ./DwC_DP/table_schemas

Cloning into 'dwc-dp'...
remote: Enumerating objects: 2757, done.
remote: Counting objects: 100% (558/558), done.
remote: Compressing objects: 100% (170/170), done.
remote: Total 2757 (delta 443), reused 432 (delta 378), pack-reused 2199 (from 1)
Receiving objects: 100% (2757/2757), 16.50 MiB | 3.92 MiB/s, done.
Resolving deltas: 100% (2079/2079), done.


In [ ]:
import json
from pathlib import Path


def combine_json_fields(input_folder, output_file):
    result = {}

    for json_file in Path(input_folder).glob("*.json"):
        with open(json_file, "r", encoding="utf-8") as f:
            data = json.load(f)

        for field in data.get("fields", []):
            name = field.get("name")

            if not name:
                continue

            result[name] = {
                "value": field.get("examples"),
                "description": field.get("description", "")
            }

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)


combine_json_fields(
    input_folder="./DwC/table-schemas",
    output_file="./DwC/dwc_dp_fields.json"
)

# GGBN Vocabulary to JSON

In [ ]:
#!/usr/bin/env python3
"""Convert the GGBN Data Standard HTML page into a JSON collection of terms.

Output shape (default):
    {
      "amplificationDate": {
        "value": "2008-10-17, May 2007",      # Examples
        "description": "Date when the amplification was carried out"   # Definition
      },
      ...
    }

Usage:
    python ggbn_html_to_terms.py page.html -o ggbn_terms.json
    python ggbn_html_to_terms.py page.html --keep-prefix --full -o ggbn_terms.json

Requires: beautifulsoup4, lxml
"""
import argparse
import json
import re
import sys

from bs4 import BeautifulSoup


def clean(text: str) -> str:
    # Collapse whitespace incl. non-breaking spaces
    return re.sub(r"\s+", " ", text.replace("\xa0", " ")).strip()


def parse_terms(html: str, keep_prefix: bool = False, full: bool = False) -> dict:
    soup = BeautifulSoup(html, "lxml")
    raw = []  # list of dicts, one per term
    current = None

    for tr in soup.find_all("tr"):
        tds = tr.find_all("td", recursive=False)
        if len(tds) != 2:
            continue
        label = clean(tds[0].get_text(" "))
        value = clean(tds[1].get_text(" "))

        if label == "Concept Name:":
            current = {"concept": value}
            raw.append(current)
        elif current is None:
            continue
        elif label == "Normative URI":
            current["uri"] = value
        elif label == "Label":
            current["label"] = value
        elif label == "Definition":
            current["definition"] = value
        elif label == "Examples":
            current["examples"] = value
        elif label == "Notes:":
            current["notes"] = value
        elif label == "":
            # "Required: No — Repeatable: Yes"
            m = re.search(r"Required:\s*(\w+).*?Repeatable:\s*(\w+)", value)
            if m:
                current["required"] = m.group(1)
                current["repeatable"] = m.group(2)

    terms = {}
    for t in raw:
        name = t["concept"]
        if not keep_prefix:
            name = name.split(":", 1)[-1]
        entry = {
            "value": t.get("examples", ""),
            "description": t.get("definition", ""),
        }
        if full:
            entry.update(
                {
                    "label": t.get("label", ""),
                    "uri": t.get("uri", ""),
                    "required": t.get("required", ""),
                    "repeatable": t.get("repeatable", ""),
                    "notes": t.get("notes", ""),
                }
            )
        # The page lists some terms under several vocabularies; identical repeats are
        # merged silently, conflicting ones are reported (first occurrence wins).
        if name in terms:
            if terms[name] != entry:
                print(f"warning: conflicting duplicate {name!r}, keeping first", file=sys.stderr)
            continue
        terms[name] = entry
    return terms

# Download HTML
url = "https://www.ggbn.org/ggbn_portal/site/wf?p=GGBN_Data_Standard_v1"

import requests

page = requests.get(url)
with open('GGBN/ggbn_page.html', 'wb+') as f:
    f.write(page.content)

# Conversion
html = "GGBN/ggbn_page.html"
ggbn_output = "GGBN/ggbn_v1_2016-10-04.json"

with open(html, encoding="utf-8") as fh:
    terms = parse_terms(fh.read(), keep_prefix=False, full=False)

out = json.dumps(terms, indent=2, ensure_ascii=False)
if ggbn_output:
    with open(ggbn_output, "w", encoding="utf-8") as fh:
        fh.write(out + "\n")
    print(f"wrote {len(terms)} terms to {ggbn_output}", file=sys.stderr)
else:
    print(out)

!rm GGBN/ggbn_page.html

wrote 139 terms to GGBN/ggbn_v1_2016-10-04.json
